# Clothes segmentation: Colab runner (Part 1, data pipeline)

This notebook only orchestrates. All logic lives in the `.py` files of the repository.

**Before running**
1. `Runtime > Change runtime type`: choose **L4 GPU** (or T4 if L4 is not offered).
2. Make the code available in one of two ways:
   - **GitHub:** push the project to a public GitHub repository and put its URL in `REPO_URL` below.
   - **Upload:** leave `REPO_URL` empty. Cell 3 asks for `clothseg_code.zip` (in the project folder on your computer) the first time and keeps a copy on Drive.
3. Run the cells from top to bottom. Every cell is safe to rerun in a new session: finished steps are skipped.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/Eyad-Elnakib1/Atelier.git"  # this repository; set to "" to use an uploaded clothseg_code.zip instead

DRIVE_ROOT = "/content/drive/MyDrive/clothseg"   # everything persistent lives here
CODE_DIR = "/content/clothes-segmentation"       # local copy of the code
HEIGHT, WIDTH = 576, 384                          # training resolution (H, W)
UPLOAD_NEW_CODE = False                           # set True once when I send you updated code, then back to False

## 2. Mount Drive and show the environment

In [ ]:
import os, platform, shutil
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

import torch, transformers, numpy, PIL, pyarrow, huggingface_hub, matplotlib
print("python      ", platform.python_version())
for m in (torch, transformers, numpy, PIL, pyarrow, huggingface_hub, matplotlib):
    print(f"{m.__name__:<16}{m.__version__}")
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: change the runtime type to a GPU")
if torch.cuda.is_available():
    print("GPU memory  ", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("CPU cores   ", os.cpu_count())
print("RAM         ", f"{os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3:.1f} GB")
print("local disk  ", f"{shutil.disk_usage('/content').free / 1024**3:.0f} GB free")
print("Drive       ", f"{shutil.disk_usage(DRIVE_ROOT).free / 1024**3:.0f} GB free")

## 3. Get the code

In [ ]:
import subprocess

if REPO_URL:
    if os.path.isdir(f"{CODE_DIR}/.git"):
        print(subprocess.run(["git", "-C", CODE_DIR, "pull"], capture_output=True, text=True).stdout)
    else:
        r = subprocess.run(["git", "clone", REPO_URL, CODE_DIR], capture_output=True, text=True)
        assert r.returncode == 0, f"git clone failed. Is the URL correct and the repository public?\n{r.stderr}"
elif os.path.isdir(f"{DRIVE_ROOT}/code") and not UPLOAD_NEW_CODE:
    shutil.copytree(f"{DRIVE_ROOT}/code", CODE_DIR, dirs_exist_ok=True)
    print("code copied from Drive")
else:
    # No code on Drive yet (or a new version was requested): upload clothseg_code.zip from your computer.
    import zipfile
    from google.colab import files
    os.chdir("/content")   # never upload into the code folder, which is replaced below
    if os.path.exists("clothseg_code.zip"):
        os.remove("clothseg_code.zip")   # otherwise Colab renames the new upload
    print("Choose clothseg_code.zip from the project folder on your computer.")
    uploaded = files.upload()
    assert "clothseg_code.zip" in uploaded, f"Expected a file named clothseg_code.zip, got {list(uploaded)}."
    for folder in (CODE_DIR, f"{DRIVE_ROOT}/code"):   # the Drive copy is used by later sessions
        shutil.rmtree(folder, ignore_errors=True)
        with zipfile.ZipFile("clothseg_code.zip") as z:
            z.extractall(folder)
    print("code uploaded and saved to Drive")

for needed in ("scripts/prepare_data.py", "src/clothseg/labels.py", "requirements.txt"):
    assert os.path.exists(f"{CODE_DIR}/{needed}"), f"{needed} is missing from {CODE_DIR}. Check the upload or the repository."
os.chdir(CODE_DIR)
print("code ready in", os.getcwd())

## 4. Install requirements

In [ ]:
!pip install -q -r requirements.txt
import segmentation_models_pytorch, timm
print("segmentation_models_pytorch", segmentation_models_pytorch.__version__, "| timm", timm.__version__, "| torch", torch.__version__)

## 5. Prepare the dataset

First run: downloads ATR (about 800 MB), saves it to Drive, resizes, splits, and saves the result to Drive as one archive.
Later sessions: copies that archive from Drive and extracts it.

In [ ]:
!python scripts/prepare_data.py --drive-root "{DRIVE_ROOT}" --height {HEIGHT} --width {WIDTH}

## 6. Sanity check

In [ ]:
!python scripts/sanity_check.py --out-dir "{DRIVE_ROOT}/reports" --height {HEIGHT} --width {WIDTH}

from IPython.display import Image, display
display(Image(f"{DRIVE_ROOT}/reports/sanity_overlays_15cls.png"))

## 7. GPU benchmark

Measures the training speed and memory of the four candidate models on this GPU. Takes a few minutes.

In [ ]:
!python scripts/benchmark.py --height {HEIGHT} --width {WIDTH} --out "{DRIVE_ROOT}/reports/benchmark.json"

## What to send back

Copy the full text output of cells 2, 5, 6 and 7, and say whether the overlay picture in cell 6 looks right
(colours sitting on the correct garments). If a cell fails, copy the whole error message.